# Poisson equation in 5D

$$-\Delta u = \pi^2\sum_{i=1}^{5}\cos(\pi x_i)\quad\text{on }[0,1]^5,\qquad u=\sum_i\cos(\pi x_i)\text{ on }\partial\Omega,$$

with exact solution $u=\sum_i\cos(\pi x_i)$.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import jax
import jax.numpy as jnp

import pinn
from pinn import operators as op, sampling

%matplotlib inline
jax.config.update("jax_enable_x64", True)   # double precision

In [ ]:
class Poisson5DProblem(pinn.Problem):
    """-Delta u = f on [0, 1]^5 (steady)."""

    x_min, x_max = 0.0, 1.0
    problem_name = "Poisson5D"
    ref_path = pinn.reference_path("poisson5d")

    def __init__(self, *, n_pde, n_bc, n_ic=0):
        self.n_pde, self.n_bc, self.n_ic = n_pde, n_bc, n_ic

    def residual_fns(self):
        return {"pde": self.pde_residual, "bc": self.bc_residual}

    def pde_residual(self, model, coords):
        u = lambda c: model(c)[0]
        f = jnp.pi**2 * jnp.sum(jnp.cos(jnp.pi * coords))
        return jnp.array([-op.laplacian(u, coords) - f])

    def bc_residual(self, model, coords):
        return jnp.array([model(coords)[0] - jnp.sum(jnp.cos(jnp.pi * coords))])

    def samplers(self):
        return {"pde": sampling.box(self.x_min, self.x_max, 5),
                "bc":  sampling.boundary_faces(self.x_min, self.x_max, 5)}

In [ ]:
cfg = pinn.RunConfig(
    network=lambda key: pinn.GaborNet(
        key, Poisson5DProblem, n_gabor=2**10,
        periodic_bc=False, n_inputs=5, n_outputs=1,
    ),
    n_pde=2**15, n_bc=2**14,
    residual_sketch=4000, parameter_sketch=4000,
    batch_size=2**12, probe_batch_size=2**8,
    pde_weight=1e-4, bc_weight=1.0,
    window_scale=6.0, n_probes=64,
)

In [ ]:
pinn.precompile64(Poisson5DProblem, cfg)

In [ ]:
results = pinn.train64(Poisson5DProblem, cfg)

## Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ref = pinn.load_reference("poisson5d")
x0, x1 = results["plot_x0"], results["plot_x1"]
lx, ly = results["plot_axes"]
extent = [x0[0], x0[-1], x1[0], x1[-1]]

for ch in ref.channels:
    pred = np.array(results["u_pred_plot"][ch])
    exact = np.array(ref.plot_grids[ch])
    err = np.abs(pred - exact)
    rel = np.linalg.norm(pred - exact) / np.linalg.norm(exact)

    fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, data, title, cmap in zip(
        axs, [pred, exact, err],
        [f"PINN  ${ch}$", f"reference  ${ch}$", f"abs error  (rel $\\ell_2$={rel:.2e})"],
        ["RdBu_r", "RdBu_r", "magma"],
    ):
        im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap)
        ax.set(xlabel=f"${lx}$", ylabel=f"${ly}$", title=title)
        fig.colorbar(im, ax=ax)
    plt.show()